# Ordered Logistic Regression Results: FAIR⁲ Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a Croissant dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is provided via a Croissant schema URL and includes ordered logistic regression outputs, survey responses, and metadata for rangeland management practices in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields by their `@id`. List record sets and, for each, display the `@id` and their available fields with `@id` and data types.

In [ ]:
# List all record sets with their @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets defined directly in this dataset metadata. Trying to infer record sets from schema...")
    # Try to infer record sets
    # this fallback is generic: some croissant datasets do not directly expose record_sets, but fields may still be present
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs and rs['field']:
            for fld in rs['field']:
                if hasattr(fld, 'to_dict'):
                    fldd = fld.to_dict()
                else:
                    fldd = fld
                print(f"  Field @id: {fldd.get('@id', str(fldd))}, dataType: {fldd.get('dataType', '-')}")
        print("")

# Alternate method - get record_sets from Dataset API if above is empty (most likely)
if len(record_sets) == 0:
    # Try mlcroissant API's internal record_set list
    from pprint import pprint
    print("Listing available record set names via dataset.record_set_ids...")
    rs_ids = [rs for rs in dataset.record_set_ids]
    pprint(rs_ids)
    # For each, print details
    for rs_id in rs_ids:
        print(f"\nRecordSet @id: {rs_id}")
        rset = dataset.record_set(rs_id)
        if hasattr(rset, 'fields'):
            for fld in rset.fields:
                print(f"  Field @id: {fld['@id']}, dataType: {fld.get('dataType', '-')}\n")
        else:
            print("  No fields found for this record set.")

## 3. Data Extraction
Load data from each available record set, using `@id` for both record sets and fields. This will create one `pandas.DataFrame` per record set.

In [ ]:
# Extract data from record sets
record_set_ids = list(dataset.record_set_ids)
dataframes = {}

for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# Show columns for first loaded record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Filter and process records for analysis. Select a numeric field and a grouping field by their `@id`. Operations include filtering, normalization, and groupby aggregation.

In [ ]:
# Identify a record set, a numeric field, and a group field for demonstration.
# We'll use the first available record set and attempt to pick fields with numeric and grouping potential.

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    field_ids = df.columns.tolist()
    # Try guessing the numeric and group field:
    # numeric: log likelihood, coefficient, or p-value fields are typical in regression output
    # group: ward, gender, county, or similar fields
    numeric_candidates = [c for c in field_ids if any(s in c.lower() for s in ["log_likelihood", "coef", "std", "pval", "iteration", "estimate", "value"])]
    group_candidates = [c for c in field_ids if any(s in c.lower() for s in ["ward", "county", "gender", "group"])]
    numeric_field_id = numeric_candidates[0] if numeric_candidates else field_ids[0]
    group_field_id = group_candidates[0] if group_candidates else (field_ids[1] if len(field_ids)>1 else field_ids[0])

    print(f"Using numeric field: {numeric_field_id}\nUsing group field: {group_field_id}\n")

    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    
    # Only keep records where numeric_field exceeds threshold
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (if numeric):")
    print(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        # Attempt to cast
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records (converted):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (mean of numeric columns):")
        print(grouped_df.head())
else:
    print("No record set dataframes available for EDA.")

## 5. Visualization
Visualize numeric field distribution and group-wise means (if available).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot the numeric field distribution (histogram)
if dataframes:
    plt.figure(figsize=(8, 5))
    # Use the same record set and numeric field from above cell
    data = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.hist(data.dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df is defined and multi-group, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_means = grouped_df[numeric_field_id] if numeric_field_id in grouped_df else grouped_df.iloc[:,0]
        grouped_means.plot(kind='bar', figsize=(8,5), color='orange')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a Croissant-packaged dataset using the `mlcroissant` library.

- Key metadata and context provided insight into the FAIR^2 dataset from Northern Kenya on rangeland management.
- All dataset elements (record sets, fields, columns) were referenced strictly via their `@id` for reproducibility.
- Exploration included dynamic detection of numeric and grouping fields for filtering, normalization, and aggregation, and basic visualizations were created from the processed record set data.

_Next steps_: Users may extend this workflow with domain-specific analyses, feature engineering, or advanced statistical methods as needed.